# Vanilla PEFT + TRL SFT — Comparison Without Unsloth

This notebook performs the same SFT using **only the HuggingFace stack** (PEFT + TRL + bitsandbytes), **without Unsloth**.

## Why?
- Understand **what Unsloth actually does** by seeing the underlying foundation
- Get comfortable with the HuggingFace stack (for models Unsloth doesn't support)
- A **direct comparison** of code complexity, speed, and VRAM

## Direct Comparison (Smoke Test, RTX 4070 Ti SUPER 16GB)

| Metric | Vanilla PEFT+TRL | Unsloth (`01_sft_modern`) |
|---|---|---|
| 3-step time | 9.47 sec | 13.4 sec |
| Sec/step | 3.16 | 4.47 |
| Peak VRAM | 4.84 GB | 4.22 GB |
| Train loss (3 steps) | 4.75 (full seq) | 2.30 (response-only) |
| Lines of code | ~25 | ~10 |
| Setup complexity | High (BnB config + prepare_kbit + LoraConfig) | Low (`from_pretrained` + `get_peft_model`) |

**Important:** The vanilla path has no `train_on_responses_only` Unsloth helper. TRL has the `assistant_only_loss=True` parameter, but it works differently (it requires the `{% generation %}` keyword in the chat template).

In production (60+ steps) Unsloth shows its **2x speed** + **30% less VRAM** advantage — for short runs the compilation overhead absorbs the difference.

## Contents
1. **Setup** — Vanilla imports (NO Unsloth)
2. **BitsAndBytesConfig** — manual 4-bit quantization
3. **Model + Tokenizer** — `AutoModelForCausalLM` + `prepare_model_for_kbit_training`
4. **LoraConfig** — manual PEFT
5. **Dataset** — same (Sebastian Raschka ch07)
6. **SFTTrainer** — standard TRL
7. **Training**
8. **Save**
9. **Unsloth vs Vanilla — Code Comparison**

## 1. Setup — Vanilla Imports

Note: **no `import unsloth`**. Just the HuggingFace stack.

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
import requests

print(f'torch: {torch.__version__} | cuda: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

## 2. BitsAndBytesConfig — Manual 4-bit Quantization

In Unsloth `load_in_4bit=True` is a single line — in vanilla you have to set up BitsAndBytesConfig:

| Parameter | Value | Description |
|---|---|---|
| `load_in_4bit` | True | Load in 4-bit |
| `bnb_4bit_quant_type` | `'nf4'` | NormalFloat4 — standard |
| `bnb_4bit_compute_dtype` | `bf16` | Compute precision |
| `bnb_4bit_use_double_quant` | True | ~0.4 extra bits saved |

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = 'nf4',
    bnb_4bit_compute_dtype = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

## 3. Model + Tokenizer

The vanilla path has 3 extra steps:
1. Load with `AutoModelForCausalLM.from_pretrained(quantization_config=...)`
2. `prepare_model_for_kbit_training(model)` — required for gradient checkpointing compatibility (QLoRA standard)
3. `model.config.use_cache = False` — `use_cache` conflicts with gradient checkpointing

**`attn_implementation='sdpa'`** — fallback when flash-attn is not installed. Older versions used `'eager'`.

In [ ]:
MODEL_NAME = 'unsloth/Qwen3-4B-Instruct-2507'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map = 'auto',
    dtype = torch.bfloat16,
    attn_implementation = 'sdpa',                # 'flash_attention_2' if installed
)
print(f'Model: {sum(p.numel() for p in model.parameters())/1e9:.2f}B params loaded')

# REQUIRED for QLoRA
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False                   # gradient checkpointing compat

## 4. LoraConfig — Manual PEFT

Same parameters as Unsloth's `get_peft_model`, but via PEFT's raw API:

In [ ]:
lora_config = LoraConfig(
    r = 32,
    lora_alpha = 32,
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout = 0,
    bias = 'none',
    task_type = 'CAUSAL_LM',                     # REQUIRED in PEFT (Unsloth does it automatically)
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Dataset — Sebastian Raschka ch07 (same)

Identical dataset to `01_sft_modern` so the comparison stays clean.

In [ ]:
url = 'https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch07/01_main-chapter-code/instruction-data.json'
raw = requests.get(url, timeout=30).json()

def alpaca_to_conversations(entry):
    user_text = entry['instruction']
    if entry.get('input'):
        user_text += f"\n\n{entry['input']}"
    return {'messages': [
        {'role': 'user',      'content': user_text},
        {'role': 'assistant', 'content': entry['output']},
    ]}

n = len(raw)
train_raw = raw[:int(n*0.85)]
dataset = Dataset.from_list([alpaca_to_conversations(e) for e in train_raw])
print(f'Train: {len(dataset)}')

# Format — vanilla has no Unsloth standardize_data_formats helper
# Use apply_chat_template directly
def fmt(examples):
    return {'text': [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in examples['messages']
    ]}
dataset = dataset.map(fmt, batched=True)
print('\nFirst 200 chars:')
print(dataset[0]['text'][:200])

## 6. SFTTrainer — Standard TRL

**IMPORTANT:** `train_on_responses_only` belongs to Unsloth and is not in vanilla. Alternatives:

1. **`assistant_only_loss=True` (SFTConfig)** — chat template must contain the `{% generation %}` keyword
2. **DataCollatorForCompletionOnlyLM** — manual response masking
3. **Do nothing** — full-sequence loss (used here in the smoke test, simple but suboptimal)

This notebook uses **option 3** for simplicity. In production I'd recommend `assistant_only_loss=True`.

In [ ]:
trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = 'text',
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,                          # demo; production: num_train_epochs=1
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = 'adamw_8bit',
        weight_decay = 0.001,
        lr_scheduler_type = 'linear',
        seed = 3407,
        report_to = 'none',
        bf16 = True,
        gradient_checkpointing = True,
        gradient_checkpointing_kwargs = {'use_reentrant': False},
        max_length = 2048,
        # assistant_only_loss = True,            # Qwen3 chat template supports `{% generation %}`
    ),
    processing_class = tokenizer,
)

## 7. Training

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_mem = round(gpu_stats.total_memory / 1024**3, 3)
print(f'GPU = {gpu_stats.name} | start mem = {start_mem} / {max_mem} GB')

trainer_stats = trainer.train()

used = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"\n=== RESULTS ===")
print(f"Train runtime: {trainer_stats.metrics['train_runtime']:.2f} sec")
print(f"Train loss:    {trainer_stats.metrics['train_loss']:.4f}")
print(f"Peak VRAM:     {used} GB")
print(f"Sec / step:    {trainer_stats.metrics['train_runtime']/60:.2f}")

## 8. Save

In [ ]:
# A. LoRA adapter (PEFT standard)
model.save_pretrained('vanilla_sft_lora')
tokenizer.save_pretrained('vanilla_sft_lora')
print('LoRA saved: vanilla_sft_lora/')

# B. Merged 16-bit (PEFT's merge_and_unload method)
# merged = model.merge_and_unload()
# merged.save_pretrained('vanilla_sft_merged', safe_serialization=True)
# tokenizer.save_pretrained('vanilla_sft_merged')

# Note: Unsloth's save_pretrained_merged and save_pretrained_gguf methods don't exist in vanilla.
# For GGUF, use llama.cpp convert-hf-to-gguf.py separately.

## 9. Unsloth vs Vanilla — Code Comparison

### Side-by-side Setup

**Unsloth (3 lines):**
```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/Qwen3-4B-Instruct-2507',
    max_seq_length=2048, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(model, r=32, ...)
```

**Vanilla (15+ lines):**
```python
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map='auto', dtype=torch.bfloat16, attn_implementation='sdpa',
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
lora_config = LoraConfig(r=32, ..., task_type='CAUSAL_LM')
model = get_peft_model(model, lora_config)
```

### Unsloth Advantages
1. **Less code** — 3 lines vs 15+
2. **Automatic optimizations** — Triton kernels, smart gradient checkpointing
3. **`get_chat_template` + `train_on_responses_only`** — you implement these yourself in vanilla
4. **`save_pretrained_merged` + `save_pretrained_gguf`** — vanilla requires manual llama.cpp
5. 

## Summary

1. **Same SFT, two different stacks** — pipeline logic is identical, code differs
2. **Unsloth = an optimized wrapper** — Triton kernels + helpers
3. **Vanilla = more code but more flexible** — supports every model
4. **`prepare_model_for_kbit_training` is REQUIRED** in the vanilla path (for QLoRA)
5. **`task_type='CAUSAL_LM'` is REQUIRED** in PEFT — Unsloth sets it automatically
6. **`use_cache=False`** for gradient checkpointing compatibility
7. **`train_on_responses_only` is Unsloth-specific** — try `assistant_only_loss=True` in vanilla

**Tested:** `smoke/06_vanilla_peft_sft_smoke.py` — 9.47 sec for 3 steps, 4.84 GB VRAM. Pipeline works.

## Practical Recommendation
- **Models Unsloth supports** → use Unsloth (faster + less VRAM)
- **Models Unsloth doesn't support** → vanilla path (multi-GPU, multimodal DPO, etc.)